In [3]:
# Imports
import pandas as pd
from cdsgd import DSClustering
from os.path import join
from sklearn.preprocessing import OneHotEncoder
import numpy as np

In [17]:
# Lectura de datos
PATH = "./"

# series de tiempo
output_df = pd.read_csv(join(PATH, "output_df.csv"), index_col=0, parse_dates=["datetime",])

# etiquetas kmeans por cliente
labels = pd.read_csv(join(PATH, "labels_df.csv"), index_col=0)

In [18]:
output_df_copy = output_df.copy()
output_df_copy["DOW"] = output_df_copy["datetime"].dt.dayofweek

# transformar datos serie de tiempo a matriz
data = (output_df_copy
 .merge(labels, on="key")
 .pivot_table(columns="datetime", values="value", index=["key", "label"], aggfunc="mean", fill_value=-1)
 .reset_index()
 )

In [19]:
# las llaves de la muestra
keys = data[["key"]]

# la etiqueta de kmeans es la variable a predecir
y_custom = data["label"]

In [20]:
# Ver la cantidad de clientes por cluster
data[["key", "label"]].groupby("label").agg("count")

datetime,key
label,
0,120
1,133
2,17


In [21]:
# la matriz de features es la matriz de serie de tiempo
X_custom = data.drop(columns=["label", "key"])
X_custom.columns = X_custom.columns.astype(str)

In [22]:
# agregar informacion de Comuna y Alimentador por cliente como OneHotEncoder

# lectura de datos de comuna y alimentador
key_comuna = pd.read_csv(join(PATH, "key_comuna_feeder.csv"),
                         index_col=0, dtype={"nro_medidor": str}
                         ).rename(columns={"nro_medidor": "key"})

# agregar metodo a la clase (compatibilidad)
def set_output(self, *, transform=None):
    """Set output container.

    See :ref:`sphx_glr_auto_examples_miscellaneous_plot_set_output.py`
    for an example on how to use the API.

    Parameters
    ----------
    transform : {"default", "pandas", "polars"}, default=None
        Configure output of `transform` and `fit_transform`.
y_custom: Default output format of a transformer
        - `"pandas"`: DataFrame output
        - `"polars"`: Polars output
        - `None`: Transform configuration is unchanged

        .. versionadded:: 1.4
            `"polars"` option was added.

    Returns
    -------
    self : estimator instance
        Estimator instance.
    """
    if transform is None:
        return self

    if not hasattr(self, "_sklearn_output_config"):
        self._sklearn_output_config = {}

    self._sklearn_output_config["transform"] = transform
    return self

OneHotEncoder.set_output = set_output

# crear OneHotEncoder
encoder = OneHotEncoder(sparse=False).set_output(transform="pandas")

FileNotFoundError: [Errno 2] No such file or directory: './key_comuna_feeder.csv'

In [81]:
# modificar tipo de columna
keys["key"] = keys["key"].astype(str)

/tmp/ipykernel_20043/3676680826.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  keys["key"] = keys["key"].astype(str)


In [82]:
# OneHotEncoding de comuna y feeder para cada key
# data_encoded = encoder.fit_transform(keys.merge(key_comuna, on="key", how="inner")[["comuna", "feeder"]])

In [83]:
## agregar a la matriz de caracteristicas el OneHotEncoding (simplemente unir lado a lado)
# X_custom = pd.concat([X_custom, pd.DataFrame(data_encoded)], axis=1, ignore_index=True)

In [23]:
X_custom

datetime,2023-07-01 00:00:00,2023-07-01 01:00:00,2023-07-01 02:00:00,2023-07-01 03:00:00,2023-07-01 04:00:00,2023-07-01 05:00:00,2023-07-01 06:00:00,2023-07-01 07:00:00,2023-07-01 08:00:00,2023-07-01 09:00:00,...,2023-07-30 15:00:00,2023-07-30 16:00:00,2023-07-30 17:00:00,2023-07-30 18:00:00,2023-07-30 19:00:00,2023-07-30 20:00:00,2023-07-30 21:00:00,2023-07-30 22:00:00,2023-07-30 23:00:00,2023-07-31 00:00:00
0,77.0,0.0,0.0,0.0,0.0,146.0,81.0,62.0,76.0,75.0,...,28.0,33.0,0.0,37.0,0.0,120.0,212.0,129.0,0.0,0.0
1,0.0,0.0,41.0,0.0,0.0,39.0,15.0,40.0,269.0,0.0,...,0.0,45.0,0.0,399.0,109.0,0.0,0.0,102.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,206.0,199.0,0.0,0.0,188.0,0.0,952.0,191.0,289.0,...,0.0,0.0,0.0,453.0,1116.0,254.0,0.0,0.0,0.0,182.0
4,0.0,165.0,181.0,147.0,0.0,138.0,0.0,0.0,0.0,0.0,...,459.0,309.0,613.0,0.0,734.0,0.0,1416.0,0.0,364.0,161.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
265,1701.0,82.0,840.0,60.0,31.0,0.0,0.0,0.0,7.0,75.0,...,0.0,234.0,0.0,313.0,164.0,0.0,82.0,0.0,0.0,0.0
266,53.0,61.0,50.0,57.0,56.0,56.0,0.0,135.0,0.0,75.0,...,0.0,774.0,0.0,0.0,0.0,0.0,1003.0,0.0,282.0,0.0
267,0.0,226.0,0.0,37.0,0.0,42.0,47.0,48.0,37.0,37.0,...,0.0,256.0,230.0,260.0,248.0,0.0,0.0,248.0,285.0,0.0
268,0.0,546.0,0.0,163.0,0.0,128.0,93.0,77.0,0.0,0.0,...,0.0,62.0,0.0,0.0,81.0,1787.0,1180.0,0.0,-1.0,-1.0


In [24]:
from dsgd import DSClassifierMultiQ
# crear instancia de DSClassifierMultiQ
DSC = DSClassifierMultiQ(3, min_iter=20, max_iter=200, debug_mode=True, lossfn="MSE", num_workers=0, min_dloss=1e-7)
X = X_custom.values
y = y_custom.values
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
losses, epoch, dt = DSC.fit(X, y, add_single_rules=True, single_rules_breaks=3, add_mult_rules=False, column_names=X_custom.columns)

Shape of X: (270, 721)
Shape of y: (270,)
Optimization started

Training time: 12.15s, epochs: 22

Least training loss reached: 0.222


In [30]:
DSC.print_most_important_rules(threshold=0.185)



Most important rules for class 0

	[0.196] R1789: -107.743 < 2023-07-19 15:00:00 < 177.349
			0: 0.192	1: 0.002	2: 0.006	Unc: 0.800

	[0.192] R963: 2023-07-11 00:00:00 > 347.766
			0: 0.185	1: 0.010	2: 0.006	Unc: 0.800

	[0.192] R2153: -111.113 < 2023-07-23 10:00:00 < 218.718
			0: 0.184	1: 0.015	2: 0.001	Unc: 0.800

	[0.192] R906: 188.870 < 2023-07-10 10:00:00 < 515.844
			0: 0.184	1: 0.014	2: 0.002	Unc: 0.800

	[0.190] R2722: 137.075 < 2023-07-29 08:00:00 < 368.194
			0: 0.181	1: 0.010	2: 0.009	Unc: 0.800

	[0.190] R1763: 2023-07-19 08:00:00 > 280.789
			0: 0.180	1: 0.000	2: 0.020	Unc: 0.800

	[0.189] R2738: 147.315 < 2023-07-29 12:00:00 < 410.434
			0: 0.179	1: 0.013	2: 0.008	Unc: 0.800

	[0.188] R1293: -71.529 < 2023-07-14 11:00:00 < 185.646
			0: 0.177	1: 0.004	2: 0.018	Unc: 0.800

	[0.188] R1847: 2023-07-20 05:00:00 > 322.641
			0: 0.176	1: 0.003	2: 0.020	Unc: 0.800

Most important rules for class 1

	[0.191] R516: 2023-07-06 09:00:00 < -124.744
			0: 0.011	1: 0.183	2: 0.006	Un

In [37]:
# filtrar filas y columnas para acelerar las pruebas
print("DSClustering begin")
print(f"Dimensions: {X_custom.shape}")

# Instantiate DSClustering
# Form 1 - Default instantiation with just the feature matrix
# ds1 = DSClustering(X_custom, max_iter=1000)
# Form 2 - Instantiation with a parameter to consider the most voted features
# ds2 = DSClustering(X_custom, most_voted=True, max_iter=1000)
# Form 3 - Instantiation with a numeric parameter
ds3 = DSClustering(X_custom, 3, debug_mode=True)
print(ds3.best)
# Apply the method to generate categorical rules
# ds1.generate_categorical_rules()  # Generate rules for the first instance
# ds2.generate_categorical_rules()  # Generate rules for the second instance
#ds3.generate_categorical_rules()  # Generate rules for the third instance

# Apply the predict method (internally finalizes the classification model)
# labels1 = ds1.predict()  # Predict labels using the first set of rules
# labels2 = ds2.predict()  # Predict labels using the second set of rules
#labels3 = ds3.predict()  # Predict labels using the third set of rules

# Apply the method to print the most important rules
# ds1.print_most_important_rules()  # Print rules from the first model
# ds2.print_most_important_rules()  # Print rules from the second model
#ds3.print_most_important_rules()  # Print rules from the third model

# Apply the method to print metrics
# is needed to encode y_custom to be use in this method
# ds1.metrics()  # Print metrics for the first model
# ds2.metrics()  # Print metrics for the second model
#ds3.metrics()  # Print metrics for the third model


DSClustering begin
Dimensions: (270, 721)
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 2 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0]


In [10]:
labels3

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [11]:
ds3.print_most_important_rules(threshold=0.0001)



Most important rules for Cluster 0

Most important rules for Cluster 1

Most important rules for Cluster 2
